In [0]:
%run ../utils/config_loader

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col

config = load_config('dev')
orders_path = config['storage']['landing']['orders']

df_orders = (spark.read.format("parquet").load(orders_path)
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
    .withColumn("environment", lit(config['environment']))
)

bronze_schema = config['schemas']['bronze']
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {config['catalog']}.{bronze_schema}")

bronze_table = f"{config['catalog']}.{bronze_schema}.orders_raw"
df_orders.write.format("delta").mode("overwrite").saveAsTable(bronze_table)

print(f"✅ {bronze_table}: {spark.table(bronze_table).count()} records")